# Сравнение классификаторов центросимметрии

Вход — 6 параметров ячейки и 729 значений $|F(hkl)|^2 / \langle |F|^2 \rangle$ на сетке $[-4,4]^3$, включая $(0,0,0)$. Всего 735 признаков. Среднее рассчитывается отдельно для каждой структуры по её наблюдаемым отражениям, включая нулевой индекс.

Класс 0 — нецентросимметричные структуры, класс 1 — центросимметричные. Для всех моделей используются одно разбиение и общие метрики.

## Датасет

При первом запуске собираем 100 000 структур через unncdb_dataset и сохраняем CSV. При следующих запусках читаем готовую таблицу

In [1]:
from pathlib import Path
import time

import numpy as np
import pandas as pd
from threadpoolctl import threadpool_limits
from unncdb_dataset import Calculation, Database, ReflectionGrid

target = 'is_centrosymmetric'
random_state = 42

In [2]:
path = Path('datasets/centrosymmetry_100k_intensity_000.csv')

if path.exists():
    df = pd.read_csv(path, dtype='float32')
else:
    db = Database('dbname=unncdb_v2')
    data = db.build(
        n=100_000,
        balance=False,
        seed=random_state,
        grid=ReflectionGrid.cube(4, include_zero=True),
        calculation=Calculation(quantity='intensity', normalization='mean_intensity'),
    )

    cell_columns = ['a', 'b', 'c', 'alpha', 'beta', 'gamma']
    reflection_columns = data.filter(like='I_').columns
    df = data[[*cell_columns, *reflection_columns, target]].astype('float32')

    path.parent.mkdir(exist_ok=True)
    df.to_csv(path, index=False, float_format='%.9g')

df[target] = df[target].astype('uint8')
assert df.shape == (100_000, 736)
df[target].value_counts().sort_index()

is_centrosymmetric
0    22540
1    77460
Name: count, dtype: int64

## Train, validation, test

80 000 структур — обучение, 10 000 — подбор параметров и выбор модели, 10 000 — итоговая проверка. Стратификация сохраняет доли классов. Все параметры и пороги фиксируются до оценки на тесте.

In [3]:
from sklearn.model_selection import train_test_split

train, holdout = train_test_split(
    df, test_size=0.2, stratify=df[target], random_state=random_state,
)
valid, test = train_test_split(
    holdout, test_size=0.5, stratify=holdout[target], random_state=random_state,
)

X_train = train.drop(columns=target)
y_train = train[target]
X_valid = valid.drop(columns=target)
y_valid = valid[target]
X_test = test.drop(columns=target)
y_test = test[target]

print(len(train), len(valid), len(test))

80000 10000 10000


## Предобработка и метрики

Пропуски заполняем медианами обучающей части. Для логистической регрессии и SVM дополнительно стандартизируем признаки. Деревья получают заполненные значения в исходном масштабе. LightAutoML выполняет предобработку внутри своего конвейера.

Основная метрика — balanced accuracy, средняя полнота двух классов. Дополнительно считаем accuracy, полноту каждого класса, macro-F1 и ROC AUC. Порог для вероятностей — 0,5.

In [4]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

imputer = SimpleImputer(strategy='median', keep_empty_features=True)
filled_train = imputer.fit_transform(X_train)
filled_valid = imputer.transform(X_valid)
filled_test = imputer.transform(X_test)

scaler = StandardScaler()
scaled_train = scaler.fit_transform(filled_train)
scaled_valid = scaler.transform(filled_valid)
scaled_test = scaler.transform(filled_test)

In [5]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    recall_score,
    roc_auc_score,
)


def balanced_accuracy(y_true, probabilities, sample_weight=None):
    return balanced_accuracy_score(
        y_true, np.asarray(probabilities).ravel() >= 0.5,
        sample_weight=sample_weight,
    )


def evaluate(y_true, predictions, scores):
    recalls = recall_score(y_true, predictions, labels=[0, 1], average=None)
    return {
        'Balanced accuracy': balanced_accuracy_score(y_true, predictions),
        'Accuracy': accuracy_score(y_true, predictions),
        'Recall 0': recalls[0],
        'Recall 1': recalls[1],
        'Macro F1': f1_score(y_true, predictions, average='macro'),
        'ROC AUC': roc_auc_score(y_true, scores),
    }

## Классификаторы и сетки параметров

Сравниваем логистическую регрессию, линейный SVM, RBF SVM, Extra Trees, LightGBM и CatBoost. Для каждого семейства сохраняем вариант с наибольшей balanced accuracy на валидации. Сетки ниже можно менять перед запуском.

Для RBF SVM проверяем C=0.1 и C=1 при gamma='scale'. Эти запуски на 80 000 структурах могут занимать десятки минут. cache_size задаёт объём кэша ядра в МБ.

In [6]:
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.base import clone
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import ParameterGrid
from sklearn.svm import LinearSVC, SVC

classifiers = [
    (
        'Logistic Regression',
        LogisticRegression(max_iter=5000, random_state=random_state),
        {'C': [0.1, 1, 10]},
        True,
    ),
    (
        'Linear SVM',
        LinearSVC(dual=False, max_iter=10000, random_state=random_state),
        {'C': [0.1, 1]},
        True,
    ),
    (
        'RBF SVM',
        SVC(kernel='rbf', cache_size=8192),
        {'C': [0.1, 1], 'gamma': ['scale']},
        True,
    ),
    (
        'Extra Trees',
        ExtraTreesClassifier(n_estimators=200, n_jobs=4, random_state=random_state),
        {'max_features': ['sqrt', 0.3], 'min_samples_leaf': [1, 3]},
        False,
    ),
    (
        'LightGBM',
        LGBMClassifier(n_jobs=1, verbosity=-1, random_state=random_state),
        {'n_estimators': [350, 700], 'num_leaves': [31, 64]},
        False,
    ),
    (
        'CatBoost',
        CatBoostClassifier(
            thread_count=1, verbose=False,
            allow_writing_files=False, random_seed=random_state,
        ),
        {'iterations': [350, 700], 'depth': [5, 7]},
        False,
    ),
]

In [7]:
best_models = {}
search_results = []

for name, estimator, parameter_grid, use_scaled in classifiers:
    fit_inputs = scaled_train if use_scaled else filled_train
    valid_inputs = scaled_valid if use_scaled else filled_valid

    for parameters in ParameterGrid(parameter_grid):
        print(f'Обучение {name}: {parameters}', flush=True)
        candidate = clone(estimator).set_params(**parameters)
        started = time.perf_counter()

        with threadpool_limits(limits=1):
            candidate.fit(fit_inputs, y_train)
            predictions = candidate.predict(valid_inputs)

        elapsed = time.perf_counter() - started
        score = balanced_accuracy_score(y_valid, predictions)
        search_results.append({
            'Model': name,
            'Parameters': parameters,
            'Validation balanced accuracy': score,
            'Seconds': elapsed,
        })

        if name not in best_models or score > best_models[name]['validation_score']:
            best_models[name] = {
                'model': candidate,
                'parameters': parameters,
                'validation_score': score,
                'use_scaled': use_scaled,
            }

        print(f'Validation balanced accuracy: {score:.4f}, {elapsed:.1f} s', flush=True)

search_table = pd.DataFrame(search_results)
print(search_table.round(4).to_string(index=False))

Обучение Logistic Regression: {'C': 0.1}
Validation balanced accuracy: 0.6172, 0.5 s
Обучение Logistic Regression: {'C': 1}
Validation balanced accuracy: 0.6201, 0.5 s
Обучение Logistic Regression: {'C': 10}
Validation balanced accuracy: 0.6262, 1.4 s
Обучение Linear SVM: {'C': 0.1}
Validation balanced accuracy: 0.5959, 27.8 s
Обучение Linear SVM: {'C': 1}
Validation balanced accuracy: 0.6023, 55.2 s
Обучение RBF SVM: {'C': 0.1, 'gamma': 'scale'}
Validation balanced accuracy: 0.7894, 598.1 s
Обучение RBF SVM: {'C': 1, 'gamma': 'scale'}
Validation balanced accuracy: 0.8628, 501.1 s
Обучение Extra Trees: {'max_features': 'sqrt', 'min_samples_leaf': 1}
Validation balanced accuracy: 0.8068, 16.7 s
Обучение Extra Trees: {'max_features': 'sqrt', 'min_samples_leaf': 3}
Validation balanced accuracy: 0.7844, 14.7 s
Обучение Extra Trees: {'max_features': 0.3, 'min_samples_leaf': 1}
Validation balanced accuracy: 0.9116, 120.6 s
Обучение Extra Trees: {'max_features': 0.3, 'min_samples_leaf': 3}
Va

## LightAutoML

LightAutoML участвует в сравнении как отдельный автоматизированный конвейер. Он обучает линейную модель, LightGBM, LightGBM с подбором параметров и CatBoost, затем подбирает веса ансамбля по той же валидации.

Бюджет поиска задаётся ниже. Время классических моделей зависит от размера их сеток, поэтому время поиска также включается в сравнение.

In [8]:
from lightautoml.automl.presets.tabular_presets import TabularAutoML
from lightautoml.tasks import Task

automl = TabularAutoML(
    task=Task('binary', metric=balanced_accuracy, greater_is_better=True),
    timeout=600,
    cpu_limit=1,
    general_params={'use_algos': [['linear_l2', 'lgb', 'lgb_tuned', 'cb']]},
    reader_params={'advanced_roles': False, 'random_state': random_state},
    selection_params={'mode': 0},
    tuning_params={'max_tuning_iter': 5, 'max_tuning_time': 120},
    lgb_params={
        'freeze_defaults': True,
        'default_params': {
            'num_leaves': 64, 'num_trees': 350, 'early_stopping_rounds': 50,
        },
    },
    cb_params={
        'freeze_defaults': True,
        'default_params': {'num_trees': 350, 'od_wait': 50},
    },
)

'nlp' extra dependency package 'fasttext-numpy2' isn't installed. Look at README.md in repo 'LightAutoML' for installation instructions.
'nlp' extra dependency package 'nltk' isn't installed. Look at README.md in repo 'LightAutoML' for installation instructions.
'nlp' extra dependency package 'transformers' isn't installed. Look at README.md in repo 'LightAutoML' for installation instructions.


/Users/vlpetrov/Desktop/UNN/Diploma/unncdb/models/lama/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/vlpetrov/Desktop/UNN/Diploma/unncdb/models/lama/.venv/lib/python3.11/site-packages/lightautoml/ml_algo/dl_model.py:40: UserWarning: 'transformers' - package isn't installed
  warnings.warn("'transformers' - package isn't installed")
/Users/vlpetrov/Desktop/UNN/Diploma/unncdb/models/lama/.venv/lib/python3.11/site-packages/lightautoml/text/embed.py:24: UserWarning: 'transformers' - package isn't installed
  warnings.warn("'transformers' - package isn't installed")
/Users/vlpetrov/Desktop/UNN/Diploma/unncdb/models/lama/.venv/lib/python3.11/site-packages/lightautoml/text/dl_transformers.py:25: UserWarning: 'transformers' - package isn't installed
  warnings.warn("'transformers' - package isn't

In [9]:
started = time.perf_counter()

with threadpool_limits(limits=1):
    automl_valid_probabilities = automl.fit_predict(
        train, roles={'target': target}, valid_data=valid, verbose=4,
    ).data[:, 0]

automl_seconds = time.perf_counter() - started
automl_validation_score = balanced_accuracy(y_valid, automl_valid_probabilities)

print(automl.create_model_str_desc())
print(f'Validation balanced accuracy: {automl_validation_score:.4f}')

[20:11:49] Stdout logging level is DEBUG.
[20:11:49] Task: binary

[20:11:49] Start automl preset with listed constraints:
[20:11:49] - time: 600.00 seconds
[20:11:49] - CPU: 1 cores
[20:11:49] - memory: 16 GB

[20:11:49] Train data shape: (80000, 736)

[20:11:50] Layer 1 train process start. Time left 598.64 secs
[20:11:52] Linear model: C = 1e-05 score = 0.5259083258130652
[20:11:52] Linear model: C = 5e-05 score = 0.5768376659928781
[20:11:52] Linear model: C = 0.0001 score = 0.5913846022024477
[20:11:53] Linear model: C = 0.0005 score = 0.6095853119141437
[20:11:53] Linear model: C = 0.001 score = 0.6125736018315318
[20:11:53] Linear model: C = 0.005 score = 0.6142672944973632
[20:11:53] Linear model: C = 0.01 score = 0.6142672944973632
[20:11:53] Linear model: C = 0.05 score = 0.6142672944973632
[20:11:53] Lvl_0_Pipe_0_Mod_0_LinearL2 fitting and predicting completed
[20:11:53] Time left 595.50 secs

[20:11:58] Training until validation scores don't improve for 50 rounds
[20:12:10]

## Выбор модели по валидации

В общую таблицу попадают лучшие параметры каждого семейства и итоговый ансамбль LightAutoML. Выбираем модель с наибольшей валидационной balanced accuracy. Время поиска — сумма времени всех проверенных вариантов семейства.

In [10]:
validation_rows = []

for name, selected in best_models.items():
    validation_rows.append({
        'Model': name,
        'Parameters': selected['parameters'],
        'Validation balanced accuracy': selected['validation_score'],
    })

validation_rows.append({
    'Model': 'LightAutoML',
    'Parameters': 'Автоматический подбор и ансамбль',
    'Validation balanced accuracy': automl_validation_score,
})

validation_table = pd.DataFrame(validation_rows).set_index('Model')
validation_table['Search seconds'] = search_table.groupby('Model')['Seconds'].sum()
validation_table.loc['LightAutoML', 'Search seconds'] = automl_seconds
validation_table = validation_table.sort_values(
    'Validation balanced accuracy', ascending=False, kind='stable',
)

best_name = validation_table.index[0]
print(validation_table.round(4).to_string())
print(f'Выбранная модель по валидации: {best_name}')

                                                       Parameters  Validation balanced accuracy  Search seconds
Model                                                                                                          
LightGBM                  {'n_estimators': 700, 'num_leaves': 64}                        0.9695        311.7344
CatBoost                          {'depth': 7, 'iterations': 700}                        0.9668        345.7794
LightAutoML                      Автоматический подбор и ансамбль                        0.9635        272.2365
Extra Trees          {'max_features': 0.3, 'min_samples_leaf': 1}                        0.9116        252.4334
RBF SVM                                {'C': 1, 'gamma': 'scale'}                        0.8628       1099.1875
Logistic Regression                                     {'C': 10}                        0.6262          2.4078
Linear SVM                                               {'C': 1}                        0.6023         

## Общая проверка на тесте

Параметры моделей уже выбраны. Оцениваем все выбранные варианты на одном тесте и добавляем их метрики в общую таблицу. Порядок строк сохраняется по валидационному качеству. Выбранная модель остаётся той, которую определили до тестирования.

In [11]:
test_rows = []
test_predictions = {}

for name, selected in best_models.items():
    candidate = selected['model']
    inputs = scaled_test if selected['use_scaled'] else filled_test

    with threadpool_limits(limits=1):
        predictions = candidate.predict(inputs).ravel()
        if hasattr(candidate, 'decision_function'):
            scores = candidate.decision_function(inputs)
        else:
            scores = candidate.predict_proba(inputs)[:, 1]

    test_predictions[name] = predictions
    test_rows.append({'Model': name, **evaluate(y_test, predictions, scores)})

with threadpool_limits(limits=1):
    automl_test_probabilities = automl.predict(X_test).data[:, 0]

test_predictions['LightAutoML'] = automl_test_probabilities >= 0.5
test_rows.append({
    'Model': 'LightAutoML',
    **evaluate(y_test, test_predictions['LightAutoML'], automl_test_probabilities),
})

test_table = pd.DataFrame(test_rows).set_index('Model')
test_table = test_table.rename(columns={
    column: f'Test {column}' for column in test_table.columns
})
result_table = validation_table.join(test_table)
print(result_table.round(4).to_string())

                                                       Parameters  Validation balanced accuracy  Search seconds  Test Balanced accuracy  Test Accuracy  Test Recall 0  Test Recall 1  Test Macro F1  Test ROC AUC
Model                                                                                                                                                                                                            
LightGBM                  {'n_estimators': 700, 'num_leaves': 64}                        0.9695        311.7344                  0.9744         0.9862         0.9530         0.9959         0.9800        0.9976
CatBoost                          {'depth': 7, 'iterations': 700}                        0.9668        345.7794                  0.9721         0.9850         0.9485         0.9956         0.9782        0.9973
LightAutoML                      Автоматический подбор и ансамбль                        0.9635        272.2365                  0.9695         0.9825         0

In [12]:
print(f'Выбранная модель: {best_name}')
print(result_table.loc[best_name].to_string())

print(classification_report(
    y_test, test_predictions[best_name], labels=[0, 1], digits=4, zero_division=0,
))

for name in result_table.index:
    print(f'\n{name}')
    print(pd.DataFrame(
        confusion_matrix(y_test, test_predictions[name], labels=[0, 1]),
        index=['True 0', 'True 1'],
        columns=['Pred 0', 'Pred 1'],
    ).to_string())

Выбранная модель: LightGBM
Parameters                      {'n_estimators': 700, 'num_leaves': 64}
Validation balanced accuracy                                   0.969476
Search seconds                                               311.734364
Test Balanced accuracy                                         0.974421
Test Accuracy                                                    0.9862
Test Recall 0                                                  0.952972
Test Recall 1                                                  0.995869
Test Macro F1                                                  0.980006
Test ROC AUC                                                   0.997632
              precision    recall  f1-score   support

           0     0.9853    0.9530    0.9689      2254
           1     0.9864    0.9959    0.9911      7746

    accuracy                         0.9862     10000
   macro avg     0.9859    0.9744    0.9800     10000
weighted avg     0.9862    0.9862    0.9861     10000